# Spatial-domain comparison panels

This notebook assembles the reported spatial-domain comparison panels for DLPFC, MERFISH hypothalamus, and STARmap MPFC. Each panel aligns manual annotations, STEP domains, and the available external-method results across sections. ARI and NMI are calculated section by section for the panel titles.

**Data source:** The panels use [DLPFC 10x Visium](https://github.com/LieberInstitute/HumanPilot), [MERFISH hypothalamus](https://datadryad.org/stash/dataset/doi:10.5061/dryad.8t8s248), and [STARmap MPFC](https://www.starmapresources.org/data).

## DLPFC Visium comparison


In [ ]:
import os

import anndata as ad
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score


def plot(adata, batch_key, label_key, axes, method, legend=None):
    """A helper function to plot spatial data for a given batch and method."""
    adata = adata[~adata.obs[label_key].isna()]
    adata.uns.pop(f"{label_key}_colors", None)
    if label_key != 'gd':
        adata.obs[label_key] = adata.obs[label_key].astype(int).astype("category")
    end = len(axes) - 1
    for i, (batch, ax) in enumerate(zip(adata.obs[batch_key].cat.categories, axes)):
        _adata = adata[adata.obs[batch_key] == batch]
        if label_key != 'gd':
            gt = _adata.obs['gd'].cat.codes
            clust = _adata.obs[label_key].cat.codes
            ari = adjusted_rand_score(gt, clust)
            nmi = normalized_mutual_info_score(gt, clust)
            title =  f"ARI: {ari:.2f}, NMI: {nmi:.2f}"
        else:
            title = batch
        # Turn off all axis labels and ticks for individual plots for a cleaner look
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xticks([])
        ax.set_yticks([])

        sc.pl.spatial(
            _adata,
            color=label_key,
            library_id=batch,
            ax=ax,
            show=False,
            title=title,
            frameon=False,
            size=1.3,
            img_key=None,
            legend_loc=None if (i < end or not legend) else 'right margin',
        )

# --- Figure and Axes Setup ---
sc.set_figure_params(figsize=(6, 4.5))
# Increased rows to 8 to accommodate Ground Truth
fig, axes = plt.subplots(nrows=8, ncols=12, figsize=(28, 21))

# --- Row 0: Ground Truth ---
method = 'Manual Annotation'
j = 0
for i, start in enumerate([151507, 151669, 151673]):
    dataset = f'DLPFC P{i+1}'
    adata = sc.read_h5ad(f"./results/DLPFC/{start}_to_{start+3}_integrated.h5ad")
    if i == 0:
        del adata.uns['gd_colors']
    adata = adata[adata.obs['gd'].cat.codes != -1]
    adata.obs['batch'] = adata.obs['batch'].astype(str).astype('category')
    plot(adata, 'batch', 'gd', axes[0][j: j+4], method, j==8)
    j += 4

# --- Row 1: STEP ---
method = 'STEP'
j = 0
for i, start in enumerate([151507, 151669, 151673]):
    dataset = f'DLPFC P{i+1}'
    adata = sc.read_h5ad(f"./results/DLPFC/{start}_to_{start+3}_integrated.h5ad")
    if i == 0:
        del adata.uns['domain_colors']
    adata = adata[adata.obs['gd'].cat.codes != -1]
    domain_key = 'cluster_' if 'cluster_' in adata.obs_keys() else 'domain'
    adata.obs[domain_key] = adata.obs[domain_key].cat.codes
    adata.obs['batch'] = adata.obs['batch'].astype(str).astype('category')
    plot(adata, 'batch', domain_key, axes[1][j: j+4], method, j==8)
    j += 4

# --- Row 2: STAligner ---
method = 'STAligner'
j = 0
for i in range(1, 4):
    dataset = f'DLPFC P{i}'
    adata = sc.read_h5ad(f"./benchmarks/staligner-res/STAligner_DLPFC_p{i}_concat.h5ad")
    adata = adata[adata.obs['gd'].cat.codes != -1]
    plot(adata, 'batch_name', 'mclust', axes[2][j: j+4], method, j==8)
    j += 4

# --- Row 3: MENDER ---
method = 'MENDER'
j = 0
for i in range(1, 4):
    dataset = f'DLPFC P{i}'
    adata = sc.read_h5ad(f"./benchmarks/mender-res/mender_dlpfc_p{i}.h5ad")
    adata = adata[adata.obs['gd'].cat.codes != -1]
    adata.obs['MENDER'] = adata.obs['MENDER'].astype(int).astype('category')
    adata.obs['batch'] = adata.obs['batch'].astype(str).astype('category')
    plot(adata, 'batch', 'MENDER', axes[3][j: j+4], method, j==8)
    j += 4

# --- Row 4: BASS ---
method = "BASS"
j = 0
for i, start in enumerate([151507, 151669, 151673]):
    dataset = f'DLPFC P{i+1}'
    adatas = {}
    for k in range(0, 4):
        batch = start + k
        adata = sc.read_h5ad(f'./data/{batch}/{batch}_annotated.h5ad')
        adatas[str(batch)] = adata
    adata = ad.concat(adatas, label='batch', uns_merge='unique')
    emb = pd.read_csv(f'./benchmarks/banksy-res/dlpfc_p{i+1}_banksy_emb.csv', index_col=0)
    labels = KMeans(n_clusters=7).fit_predict(emb)
    adata.obs['label'] = labels
    adata.obsm['emb'] = emb
    plot(adata, 'batch', 'label', axes[4][j: j+4], method, j==8)
    j += 4

# --- Row 5: GraphST ---
method = "GraphST"
j = 0
for i, start in enumerate([151507, 151669, 151673]):
    dataset = f'DLPFC P{i+1}'
    adatas = {}
    for k in range(0, 4):
        batch = start + k
        res = pd.read_csv(f'./benchmarks/graphst-res/results/Cortex_{batch}_domain.csv', index_col=0)
        adata = sc.read_h5ad(f'./data/{batch}/{batch}_annotated.h5ad')
        adata.obs['domain'] = res['domain']
        adatas[str(batch)] = adata
    adata = ad.concat(adatas, label='batch', uns_merge='unique')
    plot(adata, 'batch', 'domain', axes[5][j: j+4], method, j==8)
    j += 4

# --- Row 6: SpatialPCA ---
method = 'SpatialPCA'
j = 0
for i, start in enumerate([151507, 151669, 151673]):
    dataset = f'DLPFC P{i+1}'
    adatas = {}
    for k in range(0, 4):
        batch = start + k
        adata = sc.read_h5ad(f'./data/{batch}/{batch}_annotated.h5ad')
        label = pd.read_csv(f'./benchmarks/spatialpca-res/human_dlpfc_visium_{batch}.csv', index_col=0)
        adata.obs['label'] = label['clusterlabel_refine']
        adatas[str(batch)] = adata
    adata = ad.concat(adatas, label='batch', uns_merge='unique')
    adata = adata[~adata.obs['gd'].cat.codes != 0]
    plot(adata, 'batch', 'label', axes[6][j: j+4], method, j==8)
    j += 4

# --- Row 7: Banksy (Harmony) ---
method = 'Banksy (Harmony)'
j = 0
for i, start in enumerate([151507, 151669, 151673]):
    dataset = f'DLPFC P{i+1}'
    adatas = {}
    for k in range(0, 4):
        batch = start + k
        adata = sc.read_h5ad(f'./data/{batch}/{batch}_annotated.h5ad')
        adatas[str(batch)] = adata
    adata = ad.concat(adatas, label='batch', uns_merge='unique')
    emb = pd.read_csv(f'./benchmarks/banksy-res/dlpfc_p{i+1}_banksy_emb.csv', index_col=0)
    labels = KMeans(n_clusters=7).fit_predict(emb)
    adata.obs['label'] = labels
    adata.obsm['emb'] = emb
    adata = adata[~adata.obs['gd'].cat.codes != 0]
    plot(adata, 'batch', 'label', axes[7][j: j+4], method, j==8)
    j += 4

# --- Add Row Titles Directly to the Figure ---
ROW_TITLE_STYLE = {'rotation': 90, 'size': 12, 'weight': 'bold', 'va': 'center', 'family': 'Arial'}
method_names = [
    'Manual Annotation', 'STEP', 'STAligner', 'MENDER',
    'BASS', 'GraphST', 'SpatialPCA', 'Banksy (Harmony)'
]

for i, name in enumerate(method_names):
    # Get the position of the first subplot in each row
    ax = axes[i, 0]
    pos = ax.get_position()
    
    # Calculate the vertical center of the subplot
    y_center = (pos.y0 + pos.y1) / 2
    
    # Place the text at the calculated y-position in figure coordinates
    fig.text(0.01, y_center, name, **ROW_TITLE_STYLE)

# Use subplots_adjust to make a small amount of room on the left for the new titles
plt.subplots_adjust(left=0.04)

# Show the final figure
plt.show()


In [ ]:
fig.savefig("./benchmarks/all_dlpfc_spatial.pdf", dpi=300)


In [ ]:
fig.savefig("./benchmarks/all_dlpfc_spatial.png", dpi=300)


## MERFISH hypothalamus comparison


In [ ]:
import os
import anndata as ad
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def plot(adata, batch_key, label_key, axes, method, legend=None):
    """
    Plot one method across the sections of a spatial dataset.
    It implicitly uses 'gd' as the ground truth key for calculating metrics.
    """
    adata = adata[~adata.obs[label_key].isna()].copy()
    adata.uns.pop(f"{label_key}_colors", None)
    if label_key != 'gd':
        adata.obs[label_key] = adata.obs[label_key].astype(int).astype("category")

    end = len(axes) - 1
    for i, (batch, ax) in enumerate(zip(adata.obs[batch_key].cat.categories, axes)):
        _adata = adata[adata.obs[batch_key] == batch]

        title = f"Bregma {batch}"
        # This part requires the ground truth to be in a column named 'gd'
        if label_key != 'gd' and 'gd' in _adata.obs:
            if _adata.obs['gd'].notna().all():
                gt = _adata.obs['gd'].cat.codes
                clust = _adata.obs[label_key].cat.codes
                ari = adjusted_rand_score(gt, clust)
                nmi = normalized_mutual_info_score(gt, clust)
                title = f"ARI: {ari:.2f}, NMI: {nmi:.2f}"

        ax.set_axis_off()
        sc.pl.embedding(
            _adata, color=label_key, basis='spatial', ax=ax, show=False,
            title=title, frameon=False, size=20,
            legend_loc=None if (i < end or not legend) else 'right margin',
        )

# --- Figure and Axes Setup for MERFISH ---
sc.set_figure_params(figsize=(6, 4.5))
fig, axes = plt.subplots(nrows=8, ncols=5, figsize=(20, 24))
dataset_name = 'MERFISH Hypothalamus'

# --- Row 0: Ground Truth ---
method = 'Manual Annotation'
adata = sc.read_h5ad("./results/MERFISH/processed.h5ad")
adata.obs['gd'] = adata.obs['annotation'] # Create 'gd' column for plotter
adata.obs['Bregma'] = adata.obs['Bregma'].astype(str).astype('category')
adata.obsm['spatial'] = adata.obs[['Centroid_X', 'Centroid_Y']].values
plot(adata, 'Bregma', 'gd', axes[0], method, legend=True)


# --- Row 1: STEP ---
method = 'STEP'
adata = sc.read_h5ad(f"./results/MERFISH/processed.h5ad")
adata = adata[adata.obs['annotation'].cat.codes != -1]
adata.obs['domain'] = adata.obs['domain'].cat.codes
adata.obs['Bregma'] = adata.obs['Bregma'].astype(str).astype('category')
adata.obsm['spatial'] = adata.obs[['Centroid_X', 'Centroid_Y']].values
adata.obs['gd'] = adata.obs['annotation'] # Create 'gd' from 'annotation'
plot(adata, 'Bregma', 'domain', axes[1], method, legend=True)

# --- Row 2: STAligner ---
method = 'STAligner'
adata = sc.read_h5ad("./benchmarks/staligner-res/STAligner_merfish_animal1_concat.h5ad")
adata = adata[adata.obs['gd'].cat.codes != -1]
# 'gd' column already exists, no change needed
plot(adata, 'Bregma', 'mclust', axes[2], method, legend=True)

# --- Row 3: MENDER ---
method = 'MENDER'
adata = sc.read_h5ad(f"./benchmarks/mender-res/mender_merfish_animal1.h5ad")
adata = adata[adata.obs['gd'].cat.codes != -1]
adata.obs['MENDER'] = adata.obs['MENDER'].astype(int).astype('category')
adata.obs['Bregma'] = adata.obs['Bregma'].astype(str).astype('category')
# 'gd' column already exists, no change needed
plot(adata, 'Bregma', 'MENDER', axes[3], method, legend=True)

# --- Row 4: BASS ---
method = 'BASS'
adata = sc.read_h5ad('./data/merfish_animal1.h5ad')
for batch in os.listdir('./data/merfish_anno/'):
    anno = pd.read_csv(f'./data/merfish_anno/{batch}', index_col=0)
    batch_id = batch[:-4][6:]
    bass_label = pd.read_csv(f'./benchmarks/bass-res/merfish_{batch_id}_bass.csv')
    adata.obs.loc[adata.obs['Bregma'] == float(batch_id), 'annotation'] = anno['x'].values
    adata.obs.loc[adata.obs['Bregma'] == float(batch_id), 'bass_label'] = bass_label['bass_label'].values
adata.obs['annotation'] = adata.obs['annotation'].astype('category')
adata.obs['Bregma'] = adata.obs['Bregma'].astype(str).astype('category')
adata.obsm['spatial'] = adata.obs[['Centroid_X', 'Centroid_Y']].values
adata.obs['gd'] = adata.obs['annotation'] # Create 'gd' from 'annotation'
plot(adata, 'Bregma', 'bass_label', axes[4], method, legend=True)

# --- Row 5: GraphST ---
method = 'GraphST'
adata = sc.read_h5ad('./data/merfish_animal1.h5ad')
for batch in os.listdir('./data/merfish_anno/'):
    anno = pd.read_csv(f'./data/merfish_anno/{batch}', index_col=0)
    batch_id = batch[:-4][6:]
    bass_label = pd.read_csv(f'./benchmarks/graphst-res/results/MERFISH_Animal1_m{batch_id[1:]}_domain.csv')
    adata.obs.loc[adata.obs['Bregma'] == float(batch_id), 'annotation'] = anno['x'].values
    adata.obs.loc[adata.obs['Bregma'] == float(batch_id), 'domain'] = bass_label['domain'].values
adata.obs['annotation'] = adata.obs['annotation'].astype('category')
adata.obs['Bregma'] = adata.obs['Bregma'].astype(str).astype('category')
adata.obsm['spatial'] = adata.obs[['Centroid_X', 'Centroid_Y']].values
adata.obs['gd'] = adata.obs['annotation'] # Create 'gd' from 'annotation'
plot(adata, 'Bregma', 'domain', axes[5], method, legend=True)

# --- Row 6: SpatialPCA ---
method = 'SpatialPCA'
adata = sc.read_h5ad('./data/merfish_animal1.h5ad')
adata.obs.index = pd.Index(adata.obs['Centroid_X'].round().astype(int).astype(str) + "x" + adata.obs['Centroid_Y'].round().astype(int).astype(str))
for batch in os.listdir('./data/merfish_anno/'):
    anno = pd.read_csv(f'./data/merfish_anno/{batch}', index_col=0)
    batch_id = batch[:-4][6:]
    bass_label = pd.read_csv(f'./benchmarks/spatialpca-res/mouse_hyp_merfish_{batch_id}.csv', index_col=0)
    adata.obs.loc[adata.obs['Bregma'] == float(batch_id), 'annotation'] = anno['x'].values
    adata.obs.loc[bass_label.index, 'clusterlabel_refine'] = bass_label['clusterlabel_refine'].values
adata.obs['annotation'] = adata.obs['annotation'].astype('category')
adata.obs['Bregma'] = adata.obs['Bregma'].astype(str).astype('category')
adata.obsm['spatial'] = adata.obs[['Centroid_X', 'Centroid_Y']].values
adata.obs['gd'] = adata.obs['annotation'] # Create 'gd' from 'annotation'
plot(adata, 'Bregma', 'clusterlabel_refine', axes[6], method, legend=True)

# --- Row 7: Banksy (Harmony) ---
method = 'Banksy (Harmony)'
adata_base = sc.read_h5ad("./results/MERFISH/processed.h5ad")
adata = ad.AnnData(obs=adata_base.obs.copy(), obsm={'spatial': adata_base.obs[['Centroid_X', 'Centroid_Y']].values})
emb = pd.read_csv(f'./benchmarks/banksy-res/merfish_banksy_emb.csv', index_col=0)
labels = KMeans(n_clusters=len(adata.obs['annotation'].unique())).fit_predict(emb)
adata.obs['label'] = labels
adata.obs['Bregma'] = adata.obs['Bregma'].astype('category')
adata.obs['gd'] = adata.obs['annotation'] # Create 'gd' from 'annotation'
plot(adata, 'Bregma', 'label', axes[7], method, legend=True)

# --- Final Figure Formatting ---
ROW_TITLE_STYLE = {'rotation': 90, 'size': 14, 'weight': 'bold', 'va': 'center'}
method_names = ['Manual Annotation', 'STEP', 'STAligner', 'MENDER', 'BASS', 'GraphST', 'SpatialPCA', 'Banksy (Harmony)']
for i, name in enumerate(method_names):
    fig.text(0.015, (axes[i, 0].get_position().y0 + axes[i, 0].get_position().y1) / 2, name, **ROW_TITLE_STYLE)
plt.subplots_adjust(left=0.06, right=0.9)
fig.suptitle("MERFISH Hypothalamus Spatial Domain Comparison", fontsize=20, y=0.95)
plt.show()


In [ ]:
fig.savefig("./benchmarks/all_merfish_spatial.pdf", dpi=300)
fig.savefig("./benchmarks/all_merfish_spatial.png", dpi=300)


## STARmap MPFC comparison


In [ ]:
import os
import anndata as ad
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def plot(adata, batch_key, label_key, axes, method, legend=None):
    adata = adata[~adata.obs[label_key].isna()].copy()
    adata.uns.pop(f"{label_key}_colors", None)
    if label_key != 'gd':
        adata.obs[label_key] = adata.obs[label_key].astype(int).astype("category")

    end = len(axes) - 1
    for i, (batch, ax) in enumerate(zip(adata.obs[batch_key].cat.categories, axes)):
        _adata = adata[adata.obs[batch_key] == batch]
        title = f"Batch {i+1}"
        if label_key != 'gd' and 'gd' in _adata.obs and _adata.obs['gd'].notna().all():
            gt = _adata.obs['gd'].cat.codes
            clust = _adata.obs[label_key].cat.codes
            ari = adjusted_rand_score(gt, clust)
            nmi = normalized_mutual_info_score(gt, clust)
            title = f"ARI: {ari:.2f}, NMI: {nmi:.2f}"
        ax.set_axis_off()
        sc.pl.embedding(
            _adata, color=label_key, basis='spatial', ax=ax, show=False,
            title=title, frameon=False, size=30,
            legend_loc=None if (i < end or not legend) else 'right margin',
        )

# --- Figure and Axes Setup for STARmap ---
sc.set_figure_params(figsize=(6, 4.5))
fig, axes = plt.subplots(nrows=8, ncols=3, figsize=(12, 24))
batches = ['20180417_BZ5_control', '20180419_BZ9_control', '20180424_BZ14_control']
base_adata_path = './data/mpfc_160/merged3.h5ad'

# --- Row 0: Ground Truth ---
adata = sc.read_h5ad(base_adata_path)
plot(adata, 'batch', 'gd', axes[0], 'Manual Annotation', legend=True)

# --- Row 1: STEP ---
adata = sc.read_h5ad(f"./results/STARmap/processed.h5ad")
adata = adata[adata.obs['gd'].cat.codes != -1]
adata.obs['domain'] = adata.obs['domain'].cat.codes
adata.obs['batch'] = adata.obs['batch'].astype(str).astype('category')
plot(adata, 'batch', 'domain', axes[1], 'STEP', legend=True)

# --- Row 2: STAligner ---
adata = sc.read_h5ad("./benchmarks/staligner-res/STAligner_starmap_concat.h5ad")
adata = adata[adata.obs['gd'].cat.codes != -1]
plot(adata, 'batch', 'mclust', axes[2], 'STAligner', legend=True)

# --- Row 3: MENDER ---
adata = sc.read_h5ad(f"./benchmarks/mender-res/mender_starmap.h5ad")
adata = adata[adata.obs['gt'].cat.codes != -1]
adata.obs['MENDER'] = adata.obs['MENDER'].astype(int).astype('category')
adata.obs['batch'] = adata.obs['batch'].astype(str).astype('category')
adata.obs['gd'] = adata.obs['gt'] # Create 'gd' from 'gt' for the plotter
plot(adata, 'batch', 'MENDER', axes[3], 'MENDER', legend=True)

# --- Row 4: BASS ---
adata = sc.read_h5ad(base_adata_path)
for batch in batches:
    bass_label = pd.read_csv(f'./data/mpfc_160/starmap_{batch}_bass.csv', index_col=0)
    adata.obs.loc[adata.obs['batch'] == batch, 'bass_label'] = bass_label['bass_label']
plot(adata, 'batch', 'bass_label', axes[4], 'BASS', legend=True)

# --- Row 5: GraphST ---
adata = sc.read_h5ad(base_adata_path)
for batch in batches:
    label = pd.read_csv(f'./benchmarks/graphst-res/results/starmap_mpfc_{batch}_domain.csv', index_col=0)
    adata.obs.loc[adata.obs['batch'] == batch, 'domain'] = label['domain']
plot(adata, 'batch', 'domain', axes[5], 'GraphST', legend=True)

# --- Row 6: SpatialPCA ---
adata = sc.read_h5ad(base_adata_path)
for batch in batches:
    label = pd.read_csv(f'./benchmarks/spatialpca-res/mouse_mpfc_starmap_{batch}.csv', index_col=0)
    adata.obs.loc[adata.obs['batch'] == batch, 'clusterlabel_refine'] = label['clusterlabel_refine']
adata = adata[~adata.obs['clusterlabel_refine'].isna()]
plot(adata, 'batch', 'clusterlabel_refine', axes[6], 'SpatialPCA', legend=True)

# --- Row 7: Banksy (Harmony) ---
adata = sc.read_h5ad(base_adata_path)
emb = pd.read_csv(f'./benchmarks/banksy-res/starmap_banksy_emb.csv', index_col=0)
labels = KMeans(n_clusters=len(adata.obs['gd'].unique())).fit_predict(emb)
adata.obs['label'] = labels
plot(adata, 'batch', 'label', axes[7], 'Banksy (Harmony)', legend=True)

# --- Final Figure Formatting ---
ROW_TITLE_STYLE = {'rotation': 90, 'size': 14, 'weight': 'bold', 'va': 'center'}
method_names = ['Manual Annotation', 'STEP', 'STAligner', 'MENDER', 'BASS', 'GraphST', 'SpatialPCA', 'Banksy (Harmony)']
for i, name in enumerate(method_names):
    fig.text(0.025, (axes[i, 0].get_position().y0 + axes[i, 0].get_position().y1) / 2, name, **ROW_TITLE_STYLE)
plt.subplots_adjust(left=0.1, right=0.85)
fig.suptitle("STARmap MPFC Spatial Domain Comparison", fontsize=20, y=0.95)
plt.show()


In [ ]:
fig.savefig("./benchmarks/all_starmap_spatial.pdf", dpi=300)
fig.savefig("./benchmarks/all_starmap_spatial.png", dpi=300)
